# BM25 on MS MARCO passages

This notebook reimplements the BM25 strategy from `configs/bm25.yml` and runs it on the MS MARCO passage dataset using a `SearchStrategy`.

Config values:
- k1 = 1.2
- b = 0.75
- title_boost = 9.3
- description_boost = 4.1

MS MARCO passages only contain `description`, so the title field is empty. We still build the title index so the notebook mirrors the config.

In [ ]:
!pip install git+https://github.com/softwaredoug/cheat-at-search.git@73ae7d4cd80f
from cheat_at_search.data_dir import mount
mount(use_gdrive=True)    # colab, share data across notebook runs on gdrive
# mount(use_gdrive=False) # <- colab without gdrive
# mount(use_gdrive=False, manual_path="/path/to/directory")  # <- force data path to specific directory, ie you're running locally.


## Load MS MARCO corpus and judgments

The dataset loader will download the MS MARCO passages and small dev qrels on first use.

In [ ]:
import numpy as np
import pandas as pd

from cheat_at_search.msmarco_data import corpus, judgments

corpus = corpus.reset_index(drop=True)
doc_id_lookup = corpus["doc_id"].to_numpy()

(corpus.shape, judgments.shape)

## Implement a BM25 SearchStrategy

We build separate BM25 indices for title and description and then apply the boosts from the config when scoring.

In [ ]:
from searcharray import SearchArray
from searcharray.similarity import bm25_similarity
from cheat_at_search.tokenizers import snowball_tokenizer
from cheat_at_search.strategy.strategy import SearchStrategy

class BM25Strategy(SearchStrategy):
    def __init__(
        self,
        corpus,
        k1=1.2,
        b=0.75,
        title_boost=9.3,
        description_boost=4.1,
        top_k=10,
        workers=4,
    ):
        super().__init__(corpus, top_k=top_k, workers=workers)
        self.index = corpus
        self.title_boost = title_boost
        self.description_boost = description_boost
        self.similarity = bm25_similarity(k1=k1, b=b)

        if "title_snowball" not in self.index and "title" in self.index:
            self.index["title_snowball"] = SearchArray.index(
                self.index["title"].fillna(""),
                tokenizer=snowball_tokenizer,
                workers=workers,
            )
        if "description_snowball" not in self.index and "description" in self.index:
            self.index["description_snowball"] = SearchArray.index(
                self.index["description"].fillna(""),
                tokenizer=snowball_tokenizer,
                workers=workers,
            )

    def search(self, query, k=10):
        tokens = snowball_tokenizer(query)
        if not tokens:
            return np.array([], dtype=int), np.array([], dtype=float)

        scores = np.zeros(len(self.index), dtype=np.float32)
        if "title_snowball" in self.index:
            scores += self.title_boost * self.index["title_snowball"].array.score(
                tokens, similarity=self.similarity
            )
        if "description_snowball" in self.index:
            scores += self.description_boost * self.index["description_snowball"].array.score(
                tokens, similarity=self.similarity
            )

        top_k = min(k, len(scores))
        if top_k == 0:
            return np.array([], dtype=int), np.array([], dtype=float)

        top_idx = np.argpartition(-scores, top_k - 1)[:top_k]
        top_sorted = top_idx[np.argsort(-scores[top_idx])]
        return top_sorted, scores[top_sorted]

## Run BM25 over all queries

Scoring against the full MS MARCO collection is expensive. If you want a faster iteration loop, set `NUM_QUERIES` to a smaller number.

We disable caching in the notebook to keep the workflow simple and transparent.

In [ ]:
from cheat_at_search.search import run_strategy

TOP_K = 10
NUM_QUERIES = None  # set to e.g. 200 for faster iteration

bm25 = BM25Strategy(
    corpus,
    k1=1.2,
    b=0.75,
    title_boost=9.3,
    description_boost=4.1,
    top_k=TOP_K,
    workers=4,
)

graded = run_strategy(
    bm25,
    judgments,
    num_queries=NUM_QUERIES,
    seed=42,
    show_progress=True,
    cache=False,
)

graded.head()

## Evaluate with NDCG and MRR

In [ ]:
from cheat_at_search.search import ndcgs, mrrs

ndcg_series = ndcgs(graded)
mrr_series = mrrs(graded)

summary = pd.DataFrame(
    {
        "metric": ["NDCG@10", "MRR"],
        "value": [ndcg_series.mean(), mrr_series.mean()],
    }
)
summary

## Inspect a single query

In [ ]:
sample_query = judgments[["query"]].drop_duplicates().iloc[0]["query"]
doc_idx, scores = bm25.search(sample_query, k=5)

preview = (
    corpus.iloc[doc_idx][["doc_id", "description"]]
    .assign(score=scores)
    .reset_index(drop=True)
)
preview